In [1]:
import os
import getpass
from dotenv import load_dotenv
load_dotenv(override=True)


if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter your OpenRouter API key: ")

from langchain_openrouter import ChatOpenRouter

openrouter_model = os.getenv("MODEL", "openai/gpt-6-luna")  # swap on openrouter.ai/models if needed


/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [2]:
filename = "../../data/kdot.txt"
word_count = 0
song_count = 0

# Open the file with UTF-8 encoding to avoid Windows cp1252 decode errors
with open(filename, 'r', encoding='utf-8') as file:
    for line in file:
        if "TRACK:" in line:
            song_count += 1
        words = line.split()
        word_count += len(words)

print(f"The total number of songs in the file is: {song_count}")
print(f"The total number of words in the file is: {word_count}")

The total number of songs in the file is: 102
The total number of words in the file is: 65688


# 🔍 Retrieval in LangChain

**The problem:** A language model only knows what it learned during training. It does not know your documents, and it cannot read a 65,000-word file in one go.

**The idea: Retrieval-Augmented Generation (RAG).** Instead of relying on memory, we:
1. **Retrieve** the text most relevant to a question.
2. **Augment** the prompt with that text.
3. **Generate** an answer based on it.

Our document collection is Kendrick Lamar's lyrics (`../../data/kdot.txt`). Lyrics are dense with metaphor and emotion, so you can judge the answers yourself.

**By the end you will be able to:** load a document, split it into chunks, store the chunks as searchable vectors, and chain the steps into an answer with LCEL.

<img src="https://mintcdn.com/langchain-5e9cc07a/I6RpA28iE233vhYX/images/rag_indexing.png?w=1650&fit=max&auto=format&n=I6RpA28iE233vhYX&q=85&s=4b9e544a7a3ec168651558bce854eb60">

[LangChain Learn: RAG](https://docs.langchain.com/oss/python/langchain/rag)

In [3]:
# https://docs.langchain.com/oss/python/integrations/retrievers
# https://docs.langchain.com/oss/python/integrations/document_loaders
from langchain_community.document_loaders import TextLoader
loader = TextLoader("../../data/kdot.txt", encoding="utf-8")
kendrick_lyrics = loader.load()

/var/folders/vl/m6g01gp54cl4409b678xyrkm0000gn/T/ipykernel_39752/243565184.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
/Users/moche/IADT_Local/26_27/CC-AIEng/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [ ]:
type(kendrick_lyrics)

list

In [5]:
type(kendrick_lyrics[0])

langchain_core.documents.base.Document

## 1. Load and split

**Document loaders** read a source (a text file, a web page, a PDF) and turn it into LangChain `Document` objects. Each `Document` has `page_content` (the text) and `metadata` (where it came from). See the [loader list](https://docs.langchain.com/oss/python/integrations/document_loaders).

**Text splitters** break long documents into smaller **chunks**. We need them because:
- A model can only read a limited amount of text at once (its context window).
- Small chunks make a specific passage easier to find.

`RecursiveCharacterTextSplitter` tries to split at natural boundaries first: paragraphs, then lines, then spaces, then single characters. `chunk_size=1000` sets the maximum size (counted in characters). `chunk_overlap=50` repeats a little text between neighbouring chunks, so a sentence cut at a boundary still appears whole in one of them.

Chunk size is a trade-off. Chunks that are too small lose context. Chunks that are too large dilute the passage you need.

In [6]:
# https://docs.langchain.com/oss/python/integrations/splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap = 50,
    length_function = len,
    add_start_index = True
)

In [7]:
texts = text_splitter.split_documents(kendrick_lyrics)

In [8]:
print(texts[0])
print(texts[1])

page_content='---
ALBUM: Overly Dedicated
TRACK: Growing Apart
LYRICS:' metadata={'source': '../../data/kdot.txt', 'start_index': 0}
page_content='"Where are we going? Why are we slowing down?
Where are you going? We should be growing now
Smoke to it nigga, smoke to it nigga, oh
Where are we going? We should be slowing down
Where are you going? We should be growing now
That’s what she said to me, that place I used to call home
Is just a bed to me
And we don’t even sleep, neighbors can hear her weep
Meanwhile I’m in these streets with everybody, I’m trying to get it
And she know they got me, I watch her feelings watch me
As they staring with the saddest eyes of loneliness
Look each other in the face and barely blink
I tried to make it right, but the pen ran out of ink
So if my letters don’t reach you, I hope these lyrics in sync
That’s what it said to me
But the place I call ambition now dead to me
Gone and forgotten, I’m off track like Dale Earnhardt
My liver rotten, alcoholic tripping

## 2. Embeddings

An **embedding** is a list of numbers (a vector) that represents the meaning of a piece of text. Texts with similar meanings get vectors that sit close together.

For example, "I feel lost" and "I don't know where I'm going" share almost no words, but their embeddings should usually be close. This lets us search by meaning rather than by exact keywords.

In [15]:
from langchain_openai import OpenAIEmbeddings

# Embeddings go through OpenRouter's OpenAI-compatible endpoint, so only OPENROUTER_API_KEY is needed
embeddings = OpenAIEmbeddings(
    model="openai/text-embedding-3-small",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    check_embedding_ctx_length=False,
)

## 3. Store the vectors

A **vector store** holds the vector for every chunk and finds the chunks closest to a query's vector. This notebook uses **FAISS**, a library for fast similarity search. It keeps everything in memory, so the store disappears when the notebook restarts.

In [10]:
# https://docs.langchain.com/oss/python/integrations/vectorstores
# https://docs.langchain.com/oss/python/langchain/knowledge-base#faiss

from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(documents=texts, embedding=embeddings)

## 4. Search by meaning

`similarity_search(query)` returns the chunks whose vectors are closest to the query. Two settings matter:
- `k` sets how many results you get.
- Results are ranked by similarity, but similar does not mean correct. Always read what comes back.

In [11]:
query = "How can I practice mindfulness if I am always so busy and distracted?"

vectorstore.similarity_search(query)

[Document(id='e097660b-478a-4080-a8a2-08e9d899b08a', metadata={'source': '../../data/kdot.txt', 'start_index': 987}, page_content='Balling out but I\'m \'bout to drop the ball\nI’m calling out for help, my engineer called\nTold me come to the studio, I wanna tell him– ""Nah""\nThat’s what he said to me\nBut that place we call Heaven unfair to me, only 144 can go\nSeven billion people on planet Earth today\nIs there something that I don’t know\nI know to follow Him rather following people\nOr follow vanity cause that means I’m following evil\nGuess I’m following evil, I should follow cathedral\nBlessings I need but live like I don’t need you\nSo in conclusion\nWe all seem to stumble, planning our own demise\nForgetting the big picture and making it wallet size\nSo to what is important in my life, I apologize\nI promise to stay faithful, focused and sanctified\nWe all get distracted, the question is\nWould you bounce back or bounce backwards?\nWould you not know how to act or take action

**Read the output.** The top result is about getting distracted ("We all get distracted"), which is close to the question's wording. Another result, "I just need a little space to breathe", fits the idea of mindfulness without using that word. Not every result is useful (one is a verse about a caterpillar), so search by meaning still needs a human check.

## 5. Generate an answer

A **prompt template** is a text recipe with blanks. `{context}` is filled with the retrieved chunks and `{question}` with the user's question. The model then answers using only the context it is given.

The template asks the model to say "I don't know" when the context does not contain the answer. This can help, but it is not a guarantee. Models can still invent details.

`RetrievalQA.from_chain_type` builds the answering chain. `chain_type_kwargs` passes extra settings to it. Here, `{"prompt": QA_CHAIN_PROMPT}` replaces the default prompt with our custom one.

In [ ]:
from langchain_classic.chains import RetrievalQA

from langchain_core.prompts import PromptTemplate

template = """

Use the following pieces of context to answer the question at the end. Cite the Album name that helped shape your answer.

If you don't know the answer, just say 'Ah snap homie, I ain't gonna front. I don't know.', don't try to make up an answer.

Use three sentences maximum, relevant analogies, and keep the answer as concise as possible.

Use the active voice, and speak directly to the reader using concise language.
{context}

Question: {question}

Helpful Answer:

"""

QA_CHAIN_PROMPT = PromptTemplate.from_template(template)

llm = ChatOpenRouter(model=openrouter_model, temperature=0)

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vectorstore.as_retriever(),
    chain_type_kwargs={"prompt": QA_CHAIN_PROMPT}
)


query = "What do grief, fear, envy, and desire stem from?"


result = qa_chain.invoke({"query": query})

result['result']

'Grief, fear, envy, and desire stem from attachment to worldly things and the ego’s wants. **DAMN.** helped shape this answer by exploring fear, lust, and the struggle to let go.'

## 6. Chain the steps with LCEL

**LCEL** (LangChain Expression Language) connects steps with the `|` symbol. The output of one step becomes the input of the next.

1. `RunnableParallel` sends the question to the retriever (filling `context`) and passes the question on unchanged (filling `question`, via `RunnablePassthrough`).
2. The prompt template fills in `{context}` and `{question}`.
3. The LLM writes an answer.
4. `StrOutputParser` turns the reply into a plain string.

The same question must reach both the retriever and the prompt. Keep that in mind when you change the chain.

In [13]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

output_parser = StrOutputParser()

setup_and_retrieval = RunnableParallel(
    {"context": vectorstore.as_retriever(), "question": RunnablePassthrough()}
)
chain = setup_and_retrieval | QA_CHAIN_PROMPT | llm | output_parser

chain.invoke(query)
chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x11c0412b0>, search_kwargs={}),
  question: RunnablePassthrough()
}
| PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\n\nUse the following pieces of context to answer the question at the end. Cite the Album name that helped shape your answer.\n\nIf you don't know the answer, just say 'Ah snap homie, I ain't gonna front. I don't know.', don't try to make up an answer.\n\nUse three sentences maximum, relevant analogies, and keep the answer as concise as possible.\n\nUse the active voice, and speak directly to the reader using concise language.\n{context}\n\nQuestion: {question}\n\nHelpful Answer:\n\n")
| ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2', 'langchain-openrouter': '0.2.9'}}, client=<openrouter.sdk.OpenRouter object at 0x11f0dbcb0>, openrouter_

**In short:** a **runnable** is any LangChain step (retriever, prompt, model, parser) that you can connect to the next with `|`. `RunnableParallel` sends the question to the retriever and to `RunnablePassthrough()` at the same time. The retriever finds the `context`, and the passthrough returns the `question` unchanged, so the prompt gets both.

## 7. The same idea with helper functions

LangChain provides helpers that build the retrieve-then-answer chain for you:
- `create_stuff_documents_chain` "stuffs" all retrieved chunks into one prompt.
- `create_retrieval_chain` runs the retriever first, then passes the chunks to the stuff chain.

`create_retrieval_chain` reads the search query from the `input` key. Check which key holds your question, because an empty or wrong value means the retriever searches for the wrong thing. The result includes `context`, the chunks the model was given, so you can check what it saw.

In [ ]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

combine_docs_chain = create_stuff_documents_chain(
    llm, QA_CHAIN_PROMPT
)
retrieval_chain = create_retrieval_chain(vectorstore.as_retriever(), combine_docs_chain)


response = retrieval_chain.invoke({"input": "What is hope?", "question": "What is hope?"})
# Both keys are needed for two different reasons:
# - create_retrieval_chain reads the search query from the "input" key.
# - QA_CHAIN_PROMPT has a {question} placeholder, so the same text is also passed as "question".
# Suggested fix: change {question} to {input} in the QA_CHAIN_PROMPT template,
# then pass only {"input": "What is hope?"}.

response

{'input': 'What is hope?',
 'question': 'What is hope?',
 'context': [Document(id='39cd100a-78b6-4973-89ff-2e59f113d678', metadata={'source': '../../data/kdot.txt', 'start_index': 218036}, page_content='When y\'all came on the boat looking for hope\nAnd all you can say is that you’re looking for dope\nThese days ain’t no compromise\nAnd your pain ain’t mines half the time\nA brand new excuse ain’t shit to me\nBitch, I made my moves, with shackled feet\nCape Town\nIn today\'s day and age we practice the self pity of taking the easy way out\nYou wait on them, him and her\nBut when a blessing takes too long, that\'s when you go wrong\nYou selfish motherfucker\nWhy so sad?\nWalking around with them blue faces\nShe said I’m down on my luck\nAnd it’s something I gotta have\nBlue faces\nI hit the bank today and told them color me bad\nBlue faces\nGet that new money, and it’s breaking me down honey\nPimp-pimp (Hooray!)\n"\n---'),
  Document(id='63472f98-8a29-452d-8097-cbf2ef08b806', metadata={

## 8. Retrieve, then generate, step by step

The next cell does the same job without a helper chain. Writing each step out shows where things can go wrong:
1. Retrieve the top `k` chunks.
2. Join them into one context string.
3. Send the prompt to the model. Only now does the model see anything.

In [14]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
# Retrieve top-k documents for a question, assemble context, then call the LLM
query = "What is hope?"
# retrieve (uses the FAISS vectorstore created earlier)
docs = vectorstore.similarity_search(query, k=5)
print(f"Retrieved {len(docs)} documents.")
# assemble a context string (include a separator and any small provenance if available)
context = "\n\n---\n\n".join([getattr(d, 'page_content', str(d)) for d in docs])
print(f"Context length: {len(context)} characters")
print(f"Context preview (first 200 chars): {context[:200]}")

# build a concise prompt that includes the retrieved context and the original question
prompt_template = """Use the following pieces of context to answer the question. Cite sources when helpful.

{context}

Question: {question}

Helpful Answer (max 3 sentences):"""

filled_prompt = PromptTemplate.from_template(prompt_template)

llm = ChatOpenRouter(
    model=openrouter_model,
    temperature=0.7,
)
llm_chain = filled_prompt | llm | StrOutputParser()

print("\n--- Invoking LLM ---")
result = llm_chain.invoke({"context": context, "question": query})

# display the model output
print(f"\n--- LLM Response ---")
print(f"Result length: {len(result) if result else 0} characters")
print(f"Result: {result if result else '(empty string returned)'}")

Retrieved 5 documents.
Context length: 3625 characters
Context preview (first 200 chars): When y'all came on the boat looking for hope
And all you can say is that you’re looking for dope
These days ain’t no compromise
And your pain ain’t mines half the time
A brand new excuse ain’t shit to

--- Invoking LLM ---

--- LLM Response ---
Result length: 246 characters
Result: Hope is the choice to keep moving through pain and uncertainty, learning from what you’ve lived and giving the next generation a better chance. It’s believing in your own purpose—and in the strength of the people and community you build together.


## Limits and things to think about

- **Retrieval can miss.** If the right passage is not retrieved, the model answers from weaker context or from its own training. Check the retrieved chunks.
- **Citations may be guessed.** The prompt asks for the album name, but the album appears only in the header at the start of each track (see the output of cell 9). Most chunks do not contain it, so the model may guess.
- **Confident does not mean correct.** Language models produce fluent text whether or not the source supports it.
- **Your data leaves your computer.** Questions and chunks are sent to the model provider's API.
- **Cost and speed.** Every question sends the context to the model. More context costs more and takes longer.

## Try it yourself

- Ask a question the lyrics cannot answer (for example, "What is the capital of Ireland?"). Does the model say "I don't know"?
- Change `k` in the search or `chunk_size` in the splitter, then ask the same question. Which chunks change, and does the answer?